# Cross-model replication: non-thinking models on 2x T4 (free fallback)

Runs the paper's main experiment (GSM8K first 500 test items and MATH-500 levels 1 to 3, 238 items; SA, SC k=8, debate N3 R1, GE R2, HIERFIX M3; seeds 1 to 3; caps 640 and 1024) on **microsoft/Phi-4-mini-instruct** (GPU 0) and **Qwen/Qwen2.5-3B-Instruct** (GPU 1) in parallel, unquantized in fp16 (T4 has no bf16), with vLLM 0.9.2.

Settings before running: Accelerator **GPU T4 x2**, Internet **on** (needs a phone-verified account). No token is needed (both models are ungated). Run all cells. Expected time about 3 to 5 h; the session limit is 12 h. Records are written as they complete to `/kaggle/working/results/<model>/` (same schema as the Modal and Ollama runs) and zipped at the end; download `results_nonthinking.zip` from the Output tab, unzip into `results/cloud/` of the repository.

Resume after an interrupted session: upload the previous zip as a dataset, unzip it to `/kaggle/working/results/` before the run cell; finished units are skipped.

In [ ]:
# Kaggle's image is Python 3.13; vLLM 0.9.2 needs Python <3.13, so use a 3.11 venv via uv.
!pip install -q uv
!uv venv -q --python 3.11 /kaggle/py311
!uv pip install -q --python /kaggle/py311/bin/python vllm==0.9.2 transformers==4.53.2 numpy
!/kaggle/py311/bin/python -c "import vllm, sys; print('vllm', vllm.__version__, sys.version)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import base64, io, tarfile, os
PAYLOAD = '...'
os.makedirs('/kaggle/working/coa', exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode='r:gz').extractall('/kaggle/working/coa')
print(sorted(os.listdir('/kaggle/working/coa')))

In [ ]:
import subprocess, time, os, sys
HARD = time.time() + 11.3 * 3600  # inside the 12 h session limit
JOBS = [('phi4-mini', 'microsoft/Phi-4-mini-instruct', '0', 8000),
        ('qwen25-3b', 'Qwen/Qwen2.5-3B-Instruct', '1', 8001)]
procs = {}
for key, hf, gpu, port in JOBS:
    out = f'/kaggle/working/results/{key}'
    os.makedirs(out, exist_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, VLLM_USE_V1='0', COA_NO_PREFIX_CACHE='1')
    cmd = ['/kaggle/py311/bin/python', '-u', '/kaggle/working/coa/scripts/cloud/container_job.py',
           '--hf-id', hf, '--thinking', 'off', '--out-dir', out, '--dtype', 'half',
           '--max-model-len', '8192', '--port', str(port), '--max-num-seqs', '64',
           '--concurrency', '96', '--host', 'kaggle-t4', '--hard-deadline', str(HARD)]
    procs[key] = subprocess.Popen(cmd, env=env, stdout=open(f'{out}/job_stdout.log', 'a'),
                                  stderr=subprocess.STDOUT)
    print('started', key, procs[key].pid)

In [ ]:
# Monitor: rerun this cell to refresh (the jobs keep running).
import json, glob
for p in sorted(glob.glob('/kaggle/working/results/*/progress.json')):
    d = json.load(open(p))
    print(p.split('/')[-2], d['status'], 'units', d['units_in_files'], '/', d['units_total'],
          'tok/s', d['out_tok_per_s'], 'cap_rate', d['hit_cap_rate'], 'eta_s', d['eta_s'])
print({k: p.poll() for k, p in procs.items()})

In [ ]:
# Wait for both jobs, then zip the results for download.
for k, p in procs.items():
    print(k, 'exit', p.wait())
!cd /kaggle/working && zip -qr results_nonthinking.zip results && ls -la results_nonthinking.zip